In [0]:
from pyspark.sql.functions import col, sum, max as spark_max, avg, count, to_date

# lectura de las tablas limpias de la capa Silver
df_sensors = spark.read.table("agro_catalog.bronce.sensor_readings_silver")
df_weather = spark.read.table("agro_catalog.bronce.weather_readings_silver")
df_irrig = spark.read.table("agro_catalog.bronce.irrigation_events_silver")
df_alerts = spark.read.table("agro_catalog.bronce.alerts_silver")

# Resumen de Riego y Gasto Hidrico por Parcela data 1
print("Calculando el consumo hidrico acumulado por parcela...")

df_gold_riego = df_irrig.groupBy("plot_id").agg(
    sum("water_volume_l").alias("total_agua_consumida_litros"),
    count("event_id").alias("total_riegos_realizados"),
    avg("water_flow_l_min").alias("caudal_medio_l_min")
)

df_gold_riego.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("agro_catalog.bronce.gold_consumo_agua_parcela")
spark.sql("COMMENT ON TABLE agro_catalog.bronce.gold_consumo_agua_parcela IS 'Data Mart Gold: Metricas agregadas de consumo hidrico y caudal por parcela.'")
print("Tabla Gold 'gold_consumo_agua_parcela' creada con exito.")


# Estres Termico y Humedad Media del Suelo por Parcela data 2
print("Calculando el estres termico y humedad del suelo...")

df_gold_suelo = df_sensors.groupBy("plot_id", "crop").agg(
    spark_max("soil_temperature_c").alias("temperatura_suelo_maxima_c"),
    avg("soil_temperature_c").alias("temperatura_suelo_media_c"),
    avg("soil_moisture_pct").alias("humedad_suelo_media_pct"),
    spark_max("battery_pct").alias("bateria_max"),
    avg("battery_pct").alias("bateria_media_pct")
)

df_gold_suelo.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("agro_catalog.bronce.gold_estres_suelo")
spark.sql("COMMENT ON TABLE agro_catalog.bronce.gold_estres_suelo IS 'Data Mart Gold: Indicadores de estres termico, humedad de suelo y estado de baterias por parcela y cultivo.'")
print("Tabla Gold 'gold_estres_suelo' creada con exito.")


# Resumen Climatico Diario data 3
print("Calculando el resumen climatico diario...")

df_gold_clima = (
    df_weather
    .withColumn("fecha", to_date(col("timestamp")))
    .groupBy("fecha")
    .agg(
        spark_max("air_temperature_c").alias("temperatura_aire_maxima_c"),
        avg("air_temperature_c").alias("temperatura_aire_media_c"),
        avg("air_humidity_pct").alias("humedad_aire_media_pct"),
        sum("rainfall_mm").alias("precipitacion_total_mm"),
        avg("wind_speed_ms").alias("velocidad_viento_media_ms")
    )
)

df_gold_clima.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("agro_catalog.bronce.gold_resumen_clima_diario")
spark.sql("COMMENT ON TABLE agro_catalog.bronce.gold_resumen_clima_diario IS 'Data Mart Gold: Agregacion diaria de variables meteorologicas y precipitaciones.'")
print("Tabla Gold 'gold_resumen_clima_diario' creada con exito.")


# Resumen de Incidencias y Alertas por Parcela data 4
print("Calculando el resumen de alertas por parcela y tipo...")

df_gold_alertas = (
    df_alerts
    .groupBy("plot_id", "alert_code", "severity")
    .agg(
        count("event_id").alias("total_alertas"),
        spark_max("measured_value").alias("valor_medido_maximo"),
        spark_max("timestamp").alias("ultima_alerta_registrada")
    )
)

df_gold_alertas.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("agro_catalog.bronce.gold_resumen_alertas")
spark.sql("COMMENT ON TABLE agro_catalog.bronce.gold_resumen_alertas IS 'Data Mart Gold: Conteo de alertas operativas por parcela, codigo de alerta y severidad.'")
print("Tabla Gold 'gold_resumen_alertas' creada con exito.")


# optimizado
print("\nEjecutando compactacion OPTIMIZE en las tablas de negocio...")
spark.sql("OPTIMIZE agro_catalog.bronce.gold_consumo_agua_parcela")
spark.sql("OPTIMIZE agro_catalog.bronce.gold_estres_suelo")
spark.sql("OPTIMIZE agro_catalog.bronce.gold_resumen_clima_diario")
spark.sql("OPTIMIZE agro_catalog.bronce.gold_resumen_alertas")
print("Optimizacion completada correctamente.")

Calculando el consumo hidrico acumulado por parcela...
Tabla Gold 'gold_consumo_agua_parcela' creada con exito.
Calculando el estres termico y humedad del suelo...
Tabla Gold 'gold_estres_suelo' creada con exito.
Calculando el resumen climatico diario...
Tabla Gold 'gold_resumen_clima_diario' creada con exito.
Calculando el resumen de alertas por parcela y tipo...
Tabla Gold 'gold_resumen_alertas' creada con exito.

Ejecutando compactacion OPTIMIZE en las tablas de negocio...
Optimizacion completada correctamente.


In [0]:
import time

tablas_gold = [
    "agro_catalog.bronce.gold_consumo_agua_parcela",
    "agro_catalog.bronce.gold_estres_suelo",
    "agro_catalog.bronce.gold_resumen_clima_diario",
    "agro_catalog.bronce.gold_resumen_alertas"
]

print("--- ESTADO INICIAL (ANTES DE OPTIMIZE) ---")
for t in tablas_gold:
    spark.sql(f"DESCRIBE DETAIL {t}").select("name", "numFiles", "sizeInBytes").show(truncate=False)

# Medimos una consulta representativa ANTES
inicio = time.time()
spark.sql("SELECT * FROM agro_catalog.bronce.gold_consumo_agua_parcela ORDER BY total_agua_consumida_litros DESC").collect()
tiempo_antes = round(time.time() - inicio, 4)

# Ejecutamos la optimizacion
print("Ejecutando OPTIMIZE en las tablas Gold...")
for t in tablas_gold:
    spark.sql(f"OPTIMIZE {t}")

print("\n--- ESTADO FINAL (DESPUES DE OPTIMIZE) ---")
for t in tablas_gold:
    spark.sql(f"DESCRIBE DETAIL {t}").select("name", "numFiles", "sizeInBytes").show(truncate=False)

# Medimos la misma consulta DESPUES
inicio = time.time()
spark.sql("SELECT * FROM agro_catalog.bronce.gold_consumo_agua_parcela ORDER BY total_agua_consumida_litros DESC").collect()
tiempo_despues = round(time.time() - inicio, 4)

print(f"Tiempo consulta ANTES: {tiempo_antes}s | DESPUES: {tiempo_despues}s")

--- ESTADO INICIAL (ANTES DE OPTIMIZE) ---
+---------------------------------------------+--------+-----------+
|name                                         |numFiles|sizeInBytes|
+---------------------------------------------+--------+-----------+
|agro_catalog.bronce.gold_consumo_agua_parcela|1       |1759       |
+---------------------------------------------+--------+-----------+

+-------------------------------------+--------+-----------+
|name                                 |numFiles|sizeInBytes|
+-------------------------------------+--------+-----------+
|agro_catalog.bronce.gold_estres_suelo|1       |2563       |
+-------------------------------------+--------+-----------+

+---------------------------------------------+--------+-----------+
|name                                         |numFiles|sizeInBytes|
+---------------------------------------------+--------+-----------+
|agro_catalog.bronce.gold_resumen_clima_diario|1       |2246       |
+----------------------------